### **Loading Partitions & Defining Feature Groups**

In [1]:
import pandas as pd
import numpy as np

# 1. Load the pre-split, leakage-free partitions
train_df = pd.read_csv("../data/processed/train_encounters.csv")
test_df = pd.read_csv("../data/processed/test_encounters.csv")

print(f"Loaded Train encounters: {train_df.shape}")
print(f"Loaded Test encounters:  {test_df.shape}")

# 2. Separate features (X) and binary target (y)
target_col = "target_readmit_30d"

# Non-predictive columns to exclude:
# - Identifiers (encounter_id, patient_nbr)
# - Original multiclass label (readmitted)
# - Target itself (target_readmit_30d)
cols_to_drop = ["encounter_id", "patient_nbr", "readmitted", target_col]

X_train = train_df.drop(columns=cols_to_drop)
y_train = train_df[target_col]

X_test = test_df.drop(columns=cols_to_drop)
y_test = test_df[target_col]

print(f"\nFeature matrix X_train shape: {X_train.shape}")
print(f"Feature matrix X_test shape:  {X_test.shape}")

Loaded Train encounters: (79446, 50)
Loaded Test encounters:  (19676, 50)

Feature matrix X_train shape: (79446, 46)
Feature matrix X_test shape:  (19676, 46)


### **Isolating Numerical and Categorical Pipelines**

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# 1. Explicitly identify numerical features
num_cols = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

# 2. Identify categorical features (remaining object or integer code columns)
cat_cols = [c for c in X_train.columns if c not in num_cols]

print(f"Numerical features ({len(num_cols)}): {num_cols}")
print(f"Categorical features ({len(cat_cols)}): {cat_cols[:5]} ... (+{len(cat_cols)-5} more)")

# 3. Build sub-pipelines
num_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

# 4. Combine into a unified ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_transformer, num_cols),
        ("cat", cat_transformer, cat_cols)
    ],
    remainder="drop"
)

# Test fit-transform strictly on training data
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"\nSuccessfully fitted preprocessor!")
print(f"Transformed X_train shape: {X_train_processed.shape}")
print(f"Transformed X_test shape:  {X_test_processed.shape}")

Numerical features (8): ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']
Categorical features (38): ['race', 'gender', 'age', 'admission_type_id', 'discharge_disposition_id'] ... (+33 more)

Successfully fitted preprocessor!
Transformed X_train shape: (79446, 273)
Transformed X_test shape:  (19676, 273)


### **Establishing the Logistic Regression Baseline**

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# 1. Initialize Baseline Logistic Regression with balanced weighting
log_reg = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42,
    solver="lbfgs"
)

# 2. Train baseline model on preprocessed training data
log_reg.fit(X_train_processed, y_train)

# 3. Generate predictions and predicted probabilities on held-out test set
y_pred_lr = log_reg.predict(X_test_processed)
y_prob_lr = log_reg.predict_proba(X_test_processed)[:, 1]

# 4. Calculate key metrics
lr_roc_auc = roc_auc_score(y_test, y_prob_lr)
lr_cm = confusion_matrix(y_test, y_pred_lr)

print("================ Logistic Regression Baseline Results ================")
print(f"ROC-AUC Score: {lr_roc_auc:.4f}")
print("\nConfusion Matrix [ [TN, FP], [FN, TP] ]:")
print(lr_cm)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr, target_names=["No Early Readmit (0)", "Readmitted <30d (1)"]))

================ Logistic Regression Baseline Results ================
ROC-AUC Score: 0.6711

Confusion Matrix [ [TN, FP], [FN, TP] ]:
[[12029  5430]
 [ 1000  1217]]

Classification Report:
                      precision    recall  f1-score   support

No Early Readmit (0)       0.92      0.69      0.79     17459
 Readmitted <30d (1)       0.18      0.55      0.27      2217

            accuracy                           0.67     19676
           macro avg       0.55      0.62      0.53     19676
        weighted avg       0.84      0.67      0.73     19676



Training Candidate Models (Decision Tree & Random Forest)